In [1]:
!pip install -q pyngrok fastapi uvicorn nest-asyncio aiohttp

## Running instructions
 Go to https://aipipe.org/ login and get ai pipe token
 Go to https://ngrok.com/ login in and get your token

In [2]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# secret_value_0 = user_secrets.get_secret("AI_Pipe_key")
# secret_value_1 = user_secrets.get_secret("NGROK_T")

NGROK_AUTH_TOKEN = "ngrock token here"
AI_PIPE_API_KEY = "ai_pipe secret key here"

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [3]:
import pandas as pd
import numpy as np

np.random.seed(42)

LOCALITIES = ["Sector 14", "Sector 21", "DLF Phase 2", "Cyber Hub",
              "Old Gurugram", "Sohna Road", "Sector 45", "MG Road"]
SERVICE_TYPES = ["Plumbing", "Electrical", "AC Repair", "Cleaning",
                  "Carpentry", "Pest Control", "Appliance Repair"]

def generate_synthetic_demand(n_days=365, start_date="2025-01-01", seed=42):
    rng = np.random.default_rng(seed)
    start = pd.Timestamp(start_date)
    rows = []

    base_demand = {
        "Plumbing": 5, "Electrical": 4, "AC Repair": 3, "Cleaning": 8,
        "Carpentry": 2, "Pest Control": 2, "Appliance Repair": 3,
    }

    for day_offset in range(n_days):
        date = start + pd.Timedelta(days=day_offset)
        month = date.month
        dow = date.dayofweek
        is_weekend = 1 if dow >= 5 else 0
        is_monsoon = 1 if month in [6, 7, 8, 9] else 0
        is_summer = 1 if month in [4, 5, 6] else 0
        temp_c = 20 + 15 * np.sin((month - 1) / 12 * 2 * np.pi - np.pi / 2) + rng.normal(0, 2)
        rainfall_mm = max(0, rng.normal(15 if is_monsoon else 2, 8))

        for loc_idx, loc in enumerate(LOCALITIES):
            pop_factor = 1 + loc_idx * 0.15
            for svc in SERVICE_TYPES:
                demand = base_demand[svc] * pop_factor
                demand *= (1 + 0.4 * is_weekend)
                if svc == "Plumbing":
                    demand *= (1 + 0.9 * is_monsoon + 0.02 * rainfall_mm)
                if svc == "AC Repair":
                    demand *= (1 + 1.2 * is_summer + max(0, (temp_c - 30)) * 0.05)
                if svc == "Electrical":
                    demand *= (1 + 0.5 * is_monsoon)
                if svc == "Pest Control":
                    demand *= (1 + 0.6 * is_monsoon)

                bookings = rng.poisson(max(demand, 0.1))
                rows.append({
                    "date": date, "locality": loc, "service_type": svc,
                    "day_of_week": dow, "is_weekend": is_weekend, "month": month,
                    "is_monsoon": is_monsoon, "is_summer": is_summer,
                    "temp_c": round(float(temp_c), 1), "rainfall_mm": round(float(rainfall_mm), 1),
                    "bookings": int(bookings),
                })
    return pd.DataFrame(rows)

df = generate_synthetic_demand()
df.to_csv("gig_demand_synthetic.csv", index=False)
print(f"Generated {len(df)} rows across {df['locality'].nunique()} localities "
      f"x {df['service_type'].nunique()} service types.")
df.head()

Generated 20440 rows across 8 localities x 7 service types.


,date,locality,service_type,day_of_week,is_weekend,month,is_monsoon,is_summer,temp_c,rainfall_mm,bookings
0,2025-01-01,Sector 14,Plumbing,2,0,1,0,0,5.6,0.0,6
1,2025-01-01,Sector 14,Electrical,2,0,1,0,0,5.6,0.0,6
2,2025-01-01,Sector 14,AC Repair,2,0,1,0,0,5.6,0.0,1
3,2025-01-01,Sector 14,Cleaning,2,0,1,0,0,5.6,0.0,10
4,2025-01-01,Sector 14,Carpentry,2,0,1,0,0,5.6,0.0,4


In [4]:
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

le_loc = LabelEncoder()
le_svc = LabelEncoder()
df["locality_enc"] = le_loc.fit_transform(df["locality"])
df["service_enc"] = le_svc.fit_transform(df["service_type"])

FEATURES = ["locality_enc", "service_enc", "day_of_week", "is_weekend", "month",
            "is_monsoon", "is_summer", "temp_c", "rainfall_mm"]
TARGET = "bookings"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)
print(f"Test MAE: {mae:.3f} bookings")
print(f"Test R^2: {r2:.3f}")

# Feature importance — nice slide for the judges
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nFeature importance:\n", importance)

# Persist everything the API needs: model + encoders + feature order + valid categories
bundle = {
    "model": model,
    "le_loc": le_loc,
    "le_svc": le_svc,
    "features": FEATURES,
    "localities": sorted(LOCALITIES),
    "service_types": sorted(SERVICE_TYPES),
}
joblib.dump(bundle, "demand_model.pkl")
print("\nSaved demand_model.pkl")


Test MAE: 2.104 bookings
Test R^2: 0.736

Feature importance:
 is_monsoon      0.515935
service_enc     0.173741
is_summer       0.160457
day_of_week     0.060655
locality_enc    0.041514
rainfall_mm     0.015538
month           0.012927
is_weekend      0.012455
temp_c          0.006779
dtype: float32

Saved demand_model.pkl


In [5]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import joblib
import numpy as np

bundle = joblib.load("demand_model.pkl")
model = bundle["model"]
le_loc = bundle["le_loc"]
le_svc = bundle["le_svc"]
FEATURES = bundle["features"]

app = FastAPI(title="SIH26089 Demand Forecasting API")

# Allow the HTML frontend (served from anywhere) to call this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class PredictRequest(BaseModel):
    locality: str
    service_type: str
    day_of_week: int      # 0 = Monday ... 6 = Sunday
    month: int            # 1-12
    temp_c: float
    rainfall_mm: float

@app.get("/")
def root():
    return {
        "status": "ok",
        "message": "SIH26089 demand forecasting API is running.",
        "localities": bundle["localities"],
        "service_types": bundle["service_types"],
    }

@app.post("/predict")
def predict(req: PredictRequest):
    is_weekend = 1 if req.day_of_week >= 5 else 0
    is_monsoon = 1 if req.month in [6, 7, 8, 9] else 0
    is_summer = 1 if req.month in [4, 5, 6] else 0

    try:
        loc_enc = le_loc.transform([req.locality])[0]
    except ValueError:
        return {"error": f"Unknown locality '{req.locality}'. Valid: {bundle['localities']}"}
    try:
        svc_enc = le_svc.transform([req.service_type])[0]
    except ValueError:
        return {"error": f"Unknown service_type '{req.service_type}'. Valid: {bundle['service_types']}"}

    row = np.array([[loc_enc, svc_enc, req.day_of_week, is_weekend, req.month,
                      is_monsoon, is_summer, req.temp_c, req.rainfall_mm]])
    predicted_bookings = float(model.predict(row)[0])
    predicted_bookings = max(0.0, predicted_bookings)

    # Simple workforce allocation rule-of-thumb: assume 1 worker can comfortably
    # handle ~2.5 jobs of this service type per day -> recommend headcount.
    recommended_workers = int(np.ceil(predicted_bookings / 2.5))

    return {
        "locality": req.locality,
        "service_type": req.service_type,
        "predicted_bookings": round(predicted_bookings, 2),
        "recommended_workers": recommended_workers,
    }

@app.get("/heatmap")
def heatmap(day_of_week: int = 5, month: int = 7, temp_c: float = 32.0, rainfall_mm: float = 20.0):
    """Returns predicted demand for every locality x service_type combo,
    for a given day/month/weather scenario. Powers the dashboard heatmap."""
    is_weekend = 1 if day_of_week >= 5 else 0
    is_monsoon = 1 if month in [6, 7, 8, 9] else 0
    is_summer = 1 if month in [4, 5, 6] else 0

    results = []
    for loc in bundle["localities"]:
        loc_enc = le_loc.transform([loc])[0]
        for svc in bundle["service_types"]:
            svc_enc = le_svc.transform([svc])[0]
            row = np.array([[loc_enc, svc_enc, day_of_week, is_weekend, month,
                              is_monsoon, is_summer, temp_c, rainfall_mm]])
            pred = max(0.0, float(model.predict(row)[0]))
            results.append({
                "locality": loc,
                "service_type": svc,
                "predicted_bookings": round(pred, 2),
                "recommended_workers": int(np.ceil(pred / 2.5)),
            })
    return {"scenario": {"day_of_week": day_of_week, "month": month,
                          "temp_c": temp_c, "rainfall_mm": rainfall_mm},
            "results": results}


In [6]:
import aiohttp
from pydantic import BaseModel
from typing import Optional

AI_PIPE_URL = "https://aipipe.org/openai/v1/chat/completions"

SYSTEM_PROMPT = (
    "You are the virtual assistant for Sahakar Seva, a cooperative platform "
    "connecting households and communities with local service professionals "
    "(plumbing, electrical, AC repair, cleaning, carpentry, pest control, "
    "appliance repair, and similar) — run on a cooperative model where "
    "worker-members share ownership and earnings. You ONLY answer questions "
    "about: this platform and its services, how the cooperative model works, "
    "booking or requesting a service, demand and availability, workforce "
    "scheduling, or gig-worker welfare under the cooperative.\n\n"
    "If a question is unrelated to this project (general trivia, other topics, "
    "personal advice unrelated to the platform, etc.), politely decline and "
    "steer the user back to asking about the platform, its services, or "
    "worker scheduling. Do not answer unrelated questions even if asked "
    "persistently. Answer factually and concisely, in at most a few sentences. "
    "Do not infer the user's emotional or mental state, and do not offer "
    "emotional support or advice unless explicitly asked.\n\n"
    "The user may ask in any language and may request a specific reply "
    "language — always answer in the language specified in the user message's "
    "[language: ...] tag, translating platform/technical terms naturally "
    "rather than leaving them in English."
)

class QueryRequest(BaseModel):
    question: str
    language: Optional[str] = "English"  # e.g. "English", "Hindi", "Marathi", "Tamil"

def safety_check(question: str) -> Optional[str]:
    """Keyword-based crisis check. Runs before the model is ever called."""
    crisis_keywords = ["suicide", "kill myself", "self harm", "self-harm",
                        "end my life", "can't go on", "want to die"]
    if any(word in question.lower() for word in crisis_keywords):
        return (
            "It sounds like you might be going through something serious. "
            "Please reach out to a professional as soon as you can:\n\n"
            "India: AASRA Helpline +91-9820466726 (24x7)\n"
            "USA: 988 Suicide & Crisis Lifeline\n"
            "UK: Samaritans 116 123\n\n"
            "If you're in immediate danger, please contact local emergency services."
        )
    return None

async def async_generate_answer(query: str, language: str = "English") -> str:
    user_content = f"[language: {language}] {query}"
    payload = {
        "model": "gpt-4.1-nano",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        "temperature": 0.4,
        "max_tokens": 250,
    }
    headers = {
        "Authorization": f"Bearer {AI_PIPE_API_KEY}",
        "Content-Type": "application/json",
    }
    async with aiohttp.ClientSession() as session:
        async with session.post(AI_PIPE_URL, headers=headers, json=payload) as resp:
            if resp.status != 200:
                return f"AI Pipe error {resp.status}: {await resp.text()}"
            result = await resp.json()
            return result["choices"][0]["message"]["content"]

@app.post("/query")
async def post_query(request: QueryRequest):
    crisis_response = safety_check(request.question)
    if crisis_response:
        return {"question": request.question, "answer": crisis_response, "escalated": True}
    answer = await async_generate_answer(request.question, request.language or "English")
    return {"question": request.question, "answer": answer, "escalated": False}

@app.get("/query")
async def get_query(q: str, language: str = "English"):
    crisis_response = safety_check(q)
    if crisis_response:
        return {"question": q, "answer": crisis_response, "escalated": True}
    answer = await async_generate_answer(q, language)
    return {"question": q, "answer": answer, "escalated": False}


In [ ]:
import asyncio
import nest_asyncio
import uvicorn

nest_asyncio.apply()

public_url = ngrok.connect(8000)
print("=" * 70)
print(f" Your public API URL is: {public_url}")
print(f" Paste this into the frontend's 'API URL' field, e.g.:")
print(f" {public_url}/predict")
print("=" * 70)

config = uvicorn.Config(app=app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)

loop = asyncio.get_event_loop()
loop.run_until_complete(server.serve())


INFO:     Started server process [3983]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


 Your public API URL is: NgrokTunnel: "https://scorch-citric-denial.ngrok-free.dev" -> "http://localhost:8000"
 Paste this into the frontend's 'API URL' field, e.g.:
 NgrokTunnel: "https://scorch-citric-denial.ngrok-free.dev" -> "http://localhost:8000"/predict
